# TCC — Pipeline completo no Google Colab

Notebook único do projeto: prepara GPU/Drive/GitHub, baixa dados e SAM, executa clustering/revisão via Streamlit e roda a matriz de treinamento YOLO26 Detection/Segmentation em Base, LoRA e Full fine-tune.

Antes de executar: selecione **Runtime > Change runtime type > GPU** e configure em **Secrets**: `GITHUB_TOKEN`, `ROBOFLOW_API_KEY` e, opcionalmente, `NGROK_AUTHTOKEN`.


## 1. Verificar GPU


In [ ]:
import os, sys, subprocess, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name)
    print('VRAM:', round(p.total_memory / 1024**3, 2), 'GB')
    print('Compute capability:', f'{p.major}.{p.minor}')
!nvidia-smi


## 2. Montar Google Drive e criar armazenamento persistente


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
drive.mount('/content/drive')
RUNTIME = Path('/content/drive/MyDrive/TCC_Code_Test_runtime')
for name in ['data', 'workspace', 'exports', 'weights']:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)
print('Persistência:', RUNTIME)


## 3. Clonar ou atualizar o repositório privado


In [ ]:
from pathlib import Path
REPO = Path('/content/TCC_Code_Test')
token = userdata.get('GITHUB_TOKEN')
netrc = Path.home() / '.netrc'
netrc.write_text(f'machine github.com\nlogin x-access-token\npassword {token}\n', encoding='utf-8')
netrc.chmod(0o600)
try:
    if REPO.exists():
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', 'https://github.com/ferreis/TCC_Code_Test.git', str(REPO)], check=True)
finally:
    netrc.unlink(missing_ok=True)
os.chdir(REPO)
print('Commit TCC:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 4. Instalar dependências do Colab


In [ ]:
%%capture
!pip install -r requirements-colab.txt


## 5. Conectar data/workspace/exports/weights ao Drive


In [ ]:
import shutil
for name in ['data', 'workspace', 'exports', 'weights']:
    target = RUNTIME / name
    link = REPO / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            for p in link.iterdir():
                dst = target / p.name
                if not dst.exists():
                    shutil.copytree(p, dst) if p.is_dir() else shutil.copy2(p, dst)
            shutil.rmtree(link)
        else:
            link.unlink()
    link.symlink_to(target, target_is_directory=True)
print('Drive conectado ao projeto.')


## 6. Configurar Roboflow e baixar dataset


In [ ]:
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
os.environ['ROBOFLOW_WORKSPACE'] = 'rafael-fernando-dos-reis-mecabo'
os.environ['ROBOFLOW_PROJECT'] = 'detecao-de-cetaceos-afrzs-iqey9'
os.environ['ROBOFLOW_VERSION'] = '1'
images_dir = Path('data/roboflow/train/images')
if images_dir.exists() and any(images_dir.iterdir()):
    print('Dataset já existe:', images_dir)
else:
    subprocess.run([sys.executable, 'scripts/download_roboflow.py', '--format', 'yolo26', '--output', 'data/roboflow', '--overwrite'], check=True)


## 7. Baixar SAM


In [ ]:
sam = Path('weights/sam_vit_b_01ec64.pth')
if sam.exists():
    print('SAM já existe:', sam)
else:
    subprocess.run([sys.executable, 'scripts/download_sam.py'], check=True)


## 8. Clustering das bounding boxes já revisadas — opcional

Use esta etapa quando quiser gerar os clusters a partir das boxes do Roboflow. Se o projeto já existir no Drive, a célula não recria.


In [ ]:
from tcc_discovery.box_clustering import BoxClusteringConfig, run_box_clustering
project_dir = Path('workspace/bbox_clusters_colab')
if (project_dir / 'catalog.json').exists():
    print('Projeto já existe:', project_dir)
else:
    cfg = BoxClusteringConfig(context_scale=1.8, min_crop_size_px=64, min_cluster_size=6, embedding_batch_size=8, dino_cpu_fallback_on_oom=True)
    result = run_box_clustering('data/roboflow/train', project_dir, cfg, overwrite=False, progress_cb=lambda d,t,m: print(f'[{d}/{t}] {m}') if d % 100 == 0 or d == t else None)
    print('Imagens:', len(result['images']), 'Objetos:', len(result['objects']))


## 9. Abrir a interface Streamlit — opcional

Use para upload de imagem/vídeo, classificação de clusters e revisão. Requer `NGROK_AUTHTOKEN` nos Secrets.


In [ ]:
from pyngrok import ngrok
import time
ngrok.kill()
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
streamlit_process = subprocess.Popen(['streamlit','run','app.py','--server.port','8501','--server.headless','true'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
time.sleep(3)
public_url = ngrok.connect(8501, bind_tls=True).public_url
print('Abra a interface:', public_url)
print('Streamlit PID:', streamlit_process.pid)


## 10. Encerrar Streamlit antes de treinamentos pesados

Recomendado para liberar a GPU antes da matriz YOLO26.


In [ ]:
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    print('Streamlit encerrado.')
try:
    ngrok.kill()
except Exception:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()
!nvidia-smi


# Treinamento YOLO26

Matriz principal: Detection Base, Detection LoRA, Detection Full; Segmentation Base, Segmentation LoRA, Segmentation Full. Todos começam em YOLO26n nesta primeira rodada.


## 11. Instalar/atualizar backend LoRA (YOLO-Master)


In [ ]:
!python scripts/setup_yolo_master.py --dir /content/YOLO-Master


In [ ]:
import ultralytics, importlib.util
print('Ultralytics/YOLO-Master:', ultralytics.__version__)
print('Caminho:', ultralytics.__file__)
print('LoRA disponível:', importlib.util.find_spec('ultralytics.utils.lora') is not None)


## 12. Validar a matriz experimental


In [ ]:
!python scripts/train_experiments.py --group all --list
!python scripts/train_experiments.py --group experiments --dry-run


## 13. Smoke test YOLO26n Detection

Roda somente 3 épocas. Execute antes da bateria longa.


In [ ]:
!python scripts/train_experiments.py --group smoke_tests --only detect_yolo26n_smoke


## 14. Treinamentos de Detection: Base + LoRA + Full fine-tune


In [ ]:
!python scripts/train_experiments.py --only detect_base detect_lora detect_full


## 15. Smoke test e treinamentos de Segmentation

Só executa quando `data/segmentation/data.yaml` existir.


In [ ]:
seg = Path('data/segmentation/data.yaml')
print('Dataset de segmentação pronto:', seg.exists(), '-', seg)
if seg.exists():
    subprocess.run([sys.executable, 'scripts/train_experiments.py', '--group', 'smoke_tests', '--only', 'segment_yolo26n_smoke'], check=True)
else:
    print('Ainda não há dataset de segmentação; etapa pulada.')


In [ ]:
if Path('data/segmentation/data.yaml').exists():
    subprocess.run([sys.executable, 'scripts/train_experiments.py', '--only', 'segment_base', 'segment_lora', 'segment_full'], check=True)
else:
    print('Gere primeiro data/segmentation/data.yaml para rodar Base/LoRA/Full de segmentação.')


## 16. Comparar todos os parâmetros e métricas


In [ ]:
import pandas as pd
summary = Path('workspace/training_runs/experiment_summary.csv')
if summary.exists():
    df = pd.read_csv(summary)
    display(df)
else:
    print('A tabela aparecerá após o primeiro experimento.')


## 17. Retomar treinamento após desconexão

Descomente somente o experimento que precisar continuar.


In [ ]:
# !python scripts/train_experiments.py --only detect_base --resume
# !python scripts/train_experiments.py --only detect_lora --resume
# !python scripts/train_experiments.py --only detect_full --resume
# !python scripts/train_experiments.py --only segment_base --resume
# !python scripts/train_experiments.py --only segment_lora --resume
# !python scripts/train_experiments.py --only segment_full --resume


## Resultado persistente

Os projetos, checkpoints e métricas ficam em `MyDrive/TCC_Code_Test_runtime/`, principalmente em `workspace/training_runs/`. Assim o notebook pode ser reaberto depois sem perder os arquivos já gravados.
